# Week 20: Advanced Computer Vision, 3D Vision & NeRFs
## Notebook 02: Semantic Segmentation & U-Net Architecture from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Bounding boxes are coarse approximations: a tumor or roadway lane is not rectangular.
- **Semantic Segmentation:** Classifies every individual pixel in an image into a category (e.g. Tumor vs Healthy tissue, Road vs Sidewalk).
- Standard CNN downsampling (pooling) captures abstract semantics but completely destroys high-resolution spatial boundary details.

In 2015, Ronneberger et al. invented **U-Net**:
- Symmetric **Contracting Path (Encoder)** and **Expansive Path (Decoder)**.
- **Skip Connections:** Copies high-resolution spatial feature maps directly from encoder layers across to corresponding decoder layers!
- **Industry Standard for Biomedical Image Segmentation!**

---

### 2. Mathematical Foundations: Dice Coefficient Loss
In segmentation, positive target pixels often occupy $< 1\%$ of the image (severe class imbalance):

$$\text{Dice Coefficient} = \frac{2 |X \cap Y|}{|X| + |Y|} = \frac{2 \sum_i p_i y_i}{\sum_i p_i + \sum_i y_i}$$

$$\mathcal{L}_{\text{Dice}} = 1 - \frac{2 \sum_i p_i y_i + \epsilon}{\sum_i p_i + \sum_i y_i + \epsilon}$$
Where $\epsilon$ prevents division by zero. Unlike Cross-Entropy, Dice Loss directly maximizes pixel boundary overlap.

In [ ]:
import numpy as np

# Step 1: Implement Soft Dice Loss from Scratch
def dice_loss(y_true, y_pred, epsilon=1e-6):
    """
    y_true: binary ground truth mask (0 or 1)
    y_pred: predicted probability mask (0.0 to 1.0)
    """
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    
    intersection = np.sum(y_true_flat * y_pred_flat)
    cardinality = np.sum(y_true_flat) + np.sum(y_pred_flat)
    
    dice_score = (2.0 * intersection + epsilon) / (cardinality + epsilon)
    return 1.0 - dice_score, dice_score

# Step 2: Test on Imbalanced Mask (16x16 image with a 3x3 tumor lesion)
grid_size = 16
true_mask = np.zeros((grid_size, grid_size))
true_mask[6:9, 6:9] = 1.0  # 9 tumor pixels out of 256 total pixels (3.5% positive!)

# Prediction A: High Overlap
pred_good = np.zeros((grid_size, grid_size))
pred_good[6:9, 6:9] = 0.85
pred_good[5, 6] = 0.30

# Prediction B: Completely Missed Mask (Predicts all zeros)
pred_missed = np.zeros((grid_size, grid_size))

loss_good, dice_good = dice_loss(true_mask, pred_good)
loss_missed, dice_missed = dice_loss(true_mask, pred_missed)

print("--- Semantic Segmentation Evaluation ---")
print(f"Total Pixels: {grid_size * grid_size} | Tumor Pixels: {int(np.sum(true_mask))}")
print(f"\nGood Prediction: Dice Score = {dice_good * 100:.1f}% | Dice Loss = {loss_good:.4f}")
print(f"Missed Prediction: Dice Score = {dice_missed * 100:.1f}% | Dice Loss = {loss_missed:.4f}")

# Cross-entropy comparison:
bce_missed = -np.mean(true_mask * np.log(1e-9) + (1 - true_mask) * np.log(1.0 - 1e-9))
print(f"\nNotice: Binary Cross Entropy for total miss is only {bce_missed:.4f} (misleadingly low!), whereas Dice Loss correctly penalizes it as 1.000!")

### 3. Step-by-Step Code Walkthrough
1. **Class Imbalance Resilience:** In a 96.5% negative background, a model predicting all zeros achieves 96.5% accuracy, but receives a catastrophic Dice score of 0.0%, correctly catching model failure.
2. **Differentiable Soft Intersection:** Multiplying continuous probabilities $p_i$ with ground truth $y_i$ enables end-to-end backpropagation in PyTorch.

---

### 4. Industry Applications & Trade-offs
- **Autonomous Driving:** Segmenting drivable road surface vs curbs and vegetation.
- **Satellite Remote Sensing:** Mapping deforestation and flood inundation zones.

---

### 5. Top Interview Questions & Answers
**Q1: Why are Skip Connections critical in U-Net?**
> **Answer:** In deep autoencoders, downsampling extracts high-level semantic features but loses exact pixel coordinates. Skip connections concatenate early encoder feature maps directly to decoder layers, providing the high-frequency spatial gradients required to reconstruct precise object boundaries.

**Q2: What is the difference between Semantic and Instance Segmentation?**
> **Answer:** Semantic segmentation labels every pixel by class without distinguishing individual object instances (all pedestrians are colored with the same label). Instance segmentation (e.g. Mask R-CNN) separates and identifies each unique individual object instance (Pedestrian 1 vs Pedestrian 2).